In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("best_data.csv")

In [4]:
df.shape

(21928, 51)

In [5]:
df.columns

Index(['Latitude_Degrees', 'Longitude_Degrees', 'Distance_to_Shore',
       'Turbidity', 'Cyclone_Frequency', 'Depth_m', 'Percent_Bleaching',
       'ClimSST', 'Temperature_Kelvin', 'Temperature_Mean',
       'Temperature_Minimum', 'Temperature_Maximum',
       'Temperature_Kelvin_Standard_Deviation', 'Windspeed', 'SSTA',
       'SSTA_Standard_Deviation', 'SSTA_Minimum', 'SSTA_Maximum',
       'SSTA_Frequency', 'SSTA_Frequency_Standard_Deviation',
       'SSTA_FrequencyMax', 'SSTA_FrequencyMean', 'SSTA_DHW',
       'SSTA_DHW_Standard_Deviation', 'SSTA_DHWMax', 'SSTA_DHWMean', 'TSA',
       'TSA_Standard_Deviation', 'TSA_Minimum', 'TSA_Maximum', 'TSA_Mean',
       'TSA_Frequency', 'TSA_Frequency_Standard_Deviation', 'TSA_FrequencyMax',
       'TSA_FrequencyMean', 'TSA_DHW', 'TSA_DHW_Standard_Deviation',
       'TSA_DHWMax', 'TSA_DHWMean', 'Month_1', 'Month_2', 'Month_3', 'Month_4',
       'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10',
       'Month_11', 'Month_12'],


# PCA

In [85]:
X = df.drop("Percent_Bleaching", axis=1)
Y = df["Percent_Bleaching"]

In [86]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [87]:
X_train_raw_month, X_test_raw_month = X_train_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy(), X_test_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy()

X_train_raw_fe, X_test_fe = X_train_raw[['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']].copy(), \
       X_test_raw[['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']].copy()


X_train_raw = X_train_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])
X_test_raw = X_test_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])

In [88]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

# SEPARATE imputer for FE — and keep as DataFrame
fe_imputer = SimpleImputer(strategy='median')
X_train_raw_fe = pd.DataFrame(
    fe_imputer.fit_transform(X_train_raw_fe),
    columns=X_train_raw_fe.columns,
    index=X_train_raw_fe.index
)
X_test_fe = pd.DataFrame(
    fe_imputer.transform(X_test_fe),
    columns=X_test_fe.columns,
    index=X_test_fe.index
)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [89]:
X_train_ss.shape

(17542, 38)

In [90]:
pca = PCA(n_components=0.75)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

In [91]:
X_train_pca_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(X_train_pca.shape[1])])
X_test_pca_df = pd.DataFrame(X_test_pca, columns=[f'PC{i+1}' for i in range(X_test_pca.shape[1])])

X_train_raw_month = X_train_raw_month.reset_index(drop=True)
X_test_raw_month = X_test_raw_month.reset_index(drop=True)

X_train_raw_fe = X_train_raw_fe.reset_index(drop=True)
X_test_fe = X_test_fe.reset_index(drop=True)

X_train_pca = pd.concat([X_train_pca_df, X_train_raw_month, X_train_raw_fe], axis=1)
X_test_pca  = pd.concat([X_test_pca_df,  X_test_raw_month,  X_test_fe],       axis=1)

In [92]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Start

In [93]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=394,
    max_depth=15,
    learning_rate=0.0829,
    min_samples_leaf=16,
    l2_regularization=0.094,
    random_state=42
)
# {'max_iter': 394, 'max_depth': 15, 'learning_rate': 0.08291466086400667, 'min_samples_leaf': 16, 'l2_regularization': 0.09425830628658521}

et_model = ExtraTreesRegressor(
    n_estimators=246,
    max_depth=34,
    min_samples_leaf=1,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)

# {'n_estimators': 246, 'max_depth': 34, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

rf_model = RandomForestRegressor(
    n_estimators=165, 
    max_depth=28, 
    min_samples_split =  2,
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1, 
    random_state=42
)
# {'n_estimators': 165, 'max_depth': 28, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}

In [94]:
voting = VotingRegressor([
    ('hgb', hgb_model),
    ('et',  et_model),
    ('rf',  rf_model)
])

# Train all
hgb_model.fit(X_train_pca, y_train)
et_model.fit(X_train_pca, y_train)
rf_model.fit(X_train_pca, y_train)
voting.fit(X_train_pca, y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('hgb', ...), ('et', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.0829
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",394
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",15
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",16


In [95]:
X_test_normal = X_test_pca.copy()

# Perturbation-aware late fusion

In [96]:
print("ClimSST")
print(X_test_normal.ClimSST.max())
print(X_test_normal.ClimSST.min())
print("-"*10, '\n', "SSTA")
print(X_test_normal.SSTA.max())
print(X_test_normal.SSTA.min())
print("-"*10, '\n', "SSTA")
print("Depth")
print(X_test_normal.Depth_m.max())
print(X_test_normal.Depth_m.min())
print("-"*10, '\n', "SSTA")
print("Lon")
print(X_test_normal.Longitude_Degrees.max())
print(X_test_normal.Longitude_Degrees.min())
print("-"*10, '\n', "SSTA")
print("Lat")
print(X_test_normal.Latitude_Degrees.max())
print(X_test_normal.Latitude_Degrees.min())

ClimSST
307.02
262.15
---------- 
 SSTA
4.82
0.01
---------- 
 SSTA
Depth
42.7
0.0
---------- 
 SSTA
Lon
179.9453
-179.4872
---------- 
 SSTA
Lat
36.75
-28.1625


In [97]:
perturbation = {
    'ClimSST':           0.5,     
    'SSTA':              0.05,     
    'Depth_m':           1,      
    'Longitude_Degrees': 0.001,    # ~111 m
    'Latitude_Degrees':  0.001,    
}

FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

In [98]:
def perturb_feature(X, feature, perturbation = perturbation):
    X_perturbed_p = X.copy()
    X_perturbed_n = X.copy()
    X_perturbed_p[feature] = X[feature] + perturbation[feature]
    X_perturbed_n[feature] = X[feature] - perturbation[feature]
    
    if feature == 'Depth_m':
        X_perturbed_p[feature] = X_perturbed_p[feature].clip(lower=0)
        X_perturbed_n[feature] = X_perturbed_n[feature].clip(lower=0)
    elif feature == 'Latitude_Degrees':
        X_perturbed_p[feature] = X_perturbed_p[feature].clip(-90, 90)
        X_perturbed_n[feature] = X_perturbed_n[feature].clip(-90, 90)
    elif feature == 'Longitude_Degrees':
        X_perturbed_p[feature] = ((X_perturbed_p[feature] + 180) % 360) - 180
        X_perturbed_n[feature] = ((X_perturbed_n[feature] + 180) % 360) - 180
    elif feature == 'ClimSST':
        X_perturbed_p[feature] = X_perturbed_p[feature].clip(lower=0)  
        X_perturbed_n[feature] = X_perturbed_n[feature].clip(lower=0)  
    
    return X_perturbed_p, X_perturbed_n

def perturbation_aware(x , models):
    pred = [m.predict(x)[0] for m in models]

    error = [[], [], []]
    for feature in FEATURES:
        p, n = perturb_feature(x, feature)
        for i in range(len(models)):
            error[i].append(abs(models[i].predict(p)[0] - pred[i]))
            error[i].append(abs(models[i].predict(n)[0] - pred[i]))

    total_error = [sum(e) for e in error] 

    inverse = [1 / e for e in total_error]
    weight = [w / sum(inverse) for w in inverse]
    return sum([weight[i] * pred[i] for i in range(len(models))])

In [118]:
idx = 23
x = X_test_normal.iloc[[idx]]
models = [et_model, rf_model, hgb_model]
model_names = ['ET', 'RF', 'HGB']

# ทำซ้ำ logic ของ perturbation_aware แต่ print ออกมาด้วย
pred = [m.predict(x)[0] for m in models]
error = [[], [], []]

print("=== Error per feature per model ===")
print(f"{'Feature':<22}{'ET':>10}{'RF':>10}{'HGB':>10}")
for feature in FEATURES:
    p, n = perturb_feature(x, feature)
    row_errors = []
    for i in range(len(models)):
        err_p = abs(models[i].predict(p)[0] - pred[i])
        err_n = abs(models[i].predict(n)[0] - pred[i])
        error[i].append(err_p)
        error[i].append(err_n)
        row_errors.append(err_p + err_n)
    print(f"{feature:<22}{row_errors[0]:>10.4f}{row_errors[1]:>10.4f}{row_errors[2]:>10.4f}")

total_error = [sum(e) for e in error]
print(f"\n{'TOTAL ERROR':<22}{total_error[0]:>10.4f}{total_error[1]:>10.4f}{total_error[2]:>10.4f}")

inverse = [1 / (e + 1e-10) for e in total_error]
weight = [w / sum(inverse) for w in inverse]
print(f"{'WEIGHT':<22}{weight[0]:>10.4f}{weight[1]:>10.4f}{weight[2]:>10.4f}")

print(f"\n{'PREDICTION':<22}{pred[0]:>10.4f}{pred[1]:>10.4f}{pred[2]:>10.4f}")
print(f"\nWeighted: {sum(w*p for w,p in zip(weight, pred)):.4f}")
print(f"Actual:   4.75")

=== Error per feature per model ===
Feature                       ET        RF       HGB
Latitude_Degrees          0.0386    0.0000    0.0000
Longitude_Degrees         0.1545    0.0000    0.0000
Depth_m                   1.9087    1.3790    2.0789
ClimSST                   1.0296    0.5881    0.0000
SSTA                      0.8580    0.4993    2.0012

TOTAL ERROR               3.9895    2.4664    4.0801
WEIGHT                    0.2781    0.4499    0.2720

PREDICTION                4.7500    7.4515    9.0858

Weighted: 7.1446
Actual:   4.75


# K_fold

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingRegressor
import numpy as np
import pandas as pd

# ========================================================
# Perturbation config (ใช้กับ paper_features ที่อยู่หลัง PCA)
# ========================================================
perturbation = {
    'ClimSST':           0.5,
    'SSTA':              0.05,
    'Depth_m':           1.0,
    'Longitude_Degrees': 0.001,
    'Latitude_Degrees':  0.001,
}

FEATURES = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']


def perturb_feature_batch(X, feature, delta):
    """Perturb feature สำหรับทั้ง batch (ทุก row พร้อมกัน)"""
    X_p = X.copy()
    X_n = X.copy()
    X_p[feature] = X[feature] + delta
    X_n[feature] = X[feature] - delta
    
    if feature == 'Depth_m':
        X_p[feature] = X_p[feature].clip(lower=0)
        X_n[feature] = X_n[feature].clip(lower=0)
    elif feature == 'Latitude_Degrees':
        X_p[feature] = X_p[feature].clip(-90, 90)
        X_n[feature] = X_n[feature].clip(-90, 90)
    elif feature == 'Longitude_Degrees':
        X_p[feature] = ((X_p[feature] + 180) % 360) - 180
        X_n[feature] = ((X_n[feature] + 180) % 360) - 180
    elif feature == 'ClimSST':
        X_p[feature] = X_p[feature].clip(lower=0)
        X_n[feature] = X_n[feature].clip(lower=0)
    
    return X_p, X_n


def perturbation_aware_fusion_predict(X, models, perturbation_dict, features, eps=1e-10):
    """
    Per-sample adaptive weight ensemble (vectorized).
    
    สำหรับแต่ละ sample:
      1. คำนวณ error ของแต่ละ model ต่อการ perturb แต่ละ feature
      2. Model ไหน robust (error ต่ำ) → weight สูง
      3. ทำ weighted average ของ predictions
    
    Args:
        X: DataFrame (n_samples, n_features)
        models: list of fitted models
        perturbation_dict: dict {feature_name: delta}
        features: list of feature names ที่จะ perturb
        eps: small value to avoid division by zero
    
    Returns:
        np.ndarray: predictions (n_samples,)
    """
    n_samples = len(X)
    n_models = len(models)
    
    # --- Step 1: baseline predictions [n_models, n_samples] ---
    base_preds = np.array([m.predict(X) for m in models])
    
    # --- Step 2: คำนวณ error สะสมต่อ sample ต่อ model ---
    total_errors = np.zeros((n_models, n_samples))
    
    for feature in features:
        delta = perturbation_dict[feature]
        X_p, X_n = perturb_feature_batch(X, feature, delta)
        
        for i, model in enumerate(models):
            pred_p = model.predict(X_p)
            pred_n = model.predict(X_n)
            total_errors[i] += np.abs(pred_p - base_preds[i])
            total_errors[i] += np.abs(pred_n - base_preds[i])
    
    # --- Step 3: คำนวณ per-sample weights (inverse error) ---
    inverse = 1.0 / (total_errors + eps)        # [n_models, n_samples]
    weights = inverse / inverse.sum(axis=0)     # normalize per sample
    
    # --- Step 4: weighted average prediction ---
    final_pred = (weights * base_preds).sum(axis=0)
    return final_pred


class PerturbationAwareEnsemble:
    """
    Wrapper class ให้ ensemble นี้ใช้ interface เหมือน sklearn model
    เพื่อให้ใส่ใน CV loop ได้ง่าย
    """
    def __init__(self, models, perturbation_dict, features):
        self.models = models
        self.perturbation_dict = perturbation_dict
        self.features = features
    
    def fit(self, X, y):
        for m in self.models:
            m.fit(X, y)
        return self
    
    def predict(self, X):
        return perturbation_aware_fusion_predict(
            X, self.models, self.perturbation_dict, self.features
        )


# ========================================================
# K-Fold Cross Validation
# ========================================================
n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

month_cols = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
              'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12']
paper_features = ['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_month = X[month_cols].copy()
X_paper = X[paper_features].copy()
X_for_pca = X.drop(columns=month_cols)

# ========================================================
# Models ที่จะเปรียบเทียบ
# ========================================================
cv_models = {
    'ExtraTrees':            et_model,
    'RandomForest':          rf_model,
    'HistGradientBoost':     hgb_model,
    'Voting (Ensemble)':     VotingRegressor([
        ('et',  et_model),
        ('rf',  rf_model),
        ('hgb', hgb_model)
    ]),
    'Perturbation-Aware':    'PERTURB_AWARE',   # placeholder — handle เป็นพิเศษ
}

print("=" * 70)
print("K-Fold Cross Validation Summary (k=4)")
print("=" * 70)
print(f"{'Model':<24}{'Mean R²':>14}{'Mean RMSE':>14}{'Mean MAE':>14}")
print("-" * 70)

for name, base_model in cv_models.items():
    r2_scores, rmse_scores, mae_scores = [], [], []

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X_for_pca)):
        # ----- split -----
        X_fold_train, X_fold_test = X_for_pca.iloc[train_idx], X_for_pca.iloc[test_idx]
        month_fold_train = X_month.iloc[train_idx].reset_index(drop=True)
        month_fold_test  = X_month.iloc[test_idx].reset_index(drop=True)
        paper_fold_train = X_paper.iloc[train_idx].copy()
        paper_fold_test  = X_paper.iloc[test_idx].copy()
        y_fold_train     = Y.iloc[train_idx].reset_index(drop=True)
        y_fold_test      = Y.iloc[test_idx].reset_index(drop=True)

        # ----- impute / scale / PCA -----
        imputer     = SimpleImputer(strategy='median')
        X_train_imp = imputer.fit_transform(X_fold_train)
        X_test_imp  = imputer.transform(X_fold_test)

        fe_imputer = SimpleImputer(strategy='median')
        paper_fold_train = pd.DataFrame(
            fe_imputer.fit_transform(paper_fold_train),
            columns=paper_features
        ).reset_index(drop=True)
        paper_fold_test = pd.DataFrame(
            fe_imputer.transform(paper_fold_test),
            columns=paper_features
        ).reset_index(drop=True)

        scaler      = StandardScaler()
        X_train_ss  = scaler.fit_transform(X_train_imp)
        X_test_ss   = scaler.transform(X_test_imp)

        pca         = PCA(n_components=0.75)
        X_train_pca_fold = pca.fit_transform(X_train_ss)
        X_test_pca_fold  = pca.transform(X_test_ss)

        # ----- reattach -----
        n_pcs   = X_train_pca_fold.shape[1]
        pc_cols = [f'PC{i+1}' for i in range(n_pcs)]

        X_train_final = pd.concat([
            pd.DataFrame(X_train_pca_fold, columns=pc_cols),
            month_fold_train,
            paper_fold_train,
        ], axis=1)
        X_test_final  = pd.concat([
            pd.DataFrame(X_test_pca_fold, columns=pc_cols),
            month_fold_test,
            paper_fold_test,
        ], axis=1)

        # ----- train & predict -----
        if base_model == 'PERTURB_AWARE':
            # Train 3 base models สำหรับ ensemble นี้
            fold_et  = clone(et_model)
            fold_rf  = clone(rf_model)
            fold_hgb = clone(hgb_model)
            
            ensemble = PerturbationAwareEnsemble(
                models=[fold_et, fold_rf, fold_hgb],
                perturbation_dict=perturbation,
                features=FEATURES
            )
            ensemble.fit(X_train_final, y_fold_train)
            preds = ensemble.predict(X_test_final)
        else:
            model = clone(base_model)
            model.fit(X_train_final, y_fold_train)
            preds = model.predict(X_test_final)

        r2_scores.append(r2_score(y_fold_test, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_fold_test, preds)))
        mae_scores.append(mean_absolute_error(y_fold_test, preds))

    print(f"{name:<24}{np.mean(r2_scores):>14.4f}"
          f"{np.mean(rmse_scores):>14.4f}{np.mean(mae_scores):>14.4f}")

print("=" * 70)

K-Fold Cross Validation Summary (k=4)
Model                          Mean R²     Mean RMSE      Mean MAE
----------------------------------------------------------------------
ExtraTrees                      0.6794       12.1924        5.2899
RandomForest                    0.6584       12.5828        6.3782
HistGradientBoost               0.6192       13.2841        7.3071
Voting (Ensemble)               0.6751       12.2714        6.1767
Perturbation-Aware              0.6671       12.4214        6.2131
